# CatVTON 가상 피팅 — Colab 실행 노트북

사람 사진 + 옷 사진 → 합성 이미지. 모델은 직접 구동한다 (외부 API 미사용).

**실행 전 반드시**: 런타임 → 런타임 유형 변경 → **T4 GPU**
(P100은 최신 torch에서 Pascal 지원이 빠져 CPU로 떨어진다.)

왜 Python 3.9 환경을 따로 만드는지는 `docs/ENVIRONMENT.md` 참고.
요약: CatVTON repo에 **cp39 전용으로 컴파일된 detectron2 `.so`** 가 들어있어서
Colab 기본 Python 3.12에서는 `AutoMasker`(DensePose+SCHP) import 자체가 불가능하다.

## 1. 환경 확인

In [ ]:
!nvidia-smi
import sys, torch
print('python:', sys.version)
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
print('supported archs:', torch.cuda.get_arch_list() if torch.cuda.is_available() else '-')

## 2. 저장소 clone

결과물과 섞이지 않게 `/content/scratch` 아래에 받는다.

In [ ]:
import os

SCRATCH_DIR = '/content/scratch'
REPO_DIR = os.path.join(SCRATCH_DIR, 'CatVTON')
os.makedirs(SCRATCH_DIR, exist_ok=True)
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Zheng-Chong/CatVTON.git {REPO_DIR}
print('repo:', REPO_DIR)

## 3. Python 3.9 환경 구성

repo가 고정한 조합(Python 3.9 + torch 2.1.2)을 격리된 venv로 그대로 만든다.
노트북 커널(3.12)은 건드리지 않고, 추론만 이 venv의 인터프리터로 subprocess 실행한다.

`fvcore` 이하는 detectron2 런타임 의존성인데 **repo의 `requirements.txt`에 빠져 있어서** 직접 추가한다.

> 최초 1회 약 5~10분 소요 (torch cu121 휠이 2GB대).

In [ ]:
VENV = '/content/venv39'
PY = VENV + '/bin/python'

if not os.path.exists('/usr/bin/python3.9'):
    !add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
    !apt-get install -y -qq python3.9 python3.9-venv python3.9-dev > /dev/null 2>&1

if not os.path.exists(PY):
    !python3.9 -m venv {VENV}
    !{PY} -m pip install -q --upgrade pip

!{PY} -m pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!{PY} -m pip install -q accelerate==0.31.0 diffusers==0.29.2 huggingface_hub==0.23.4 \
    transformers==4.27.3 numpy==1.26.4 opencv-python==4.10.0.84 pillow==10.3.0 PyYAML==6.0.1 \
    scipy==1.13.1 scikit-image==0.24.0 tqdm==4.66.4 matplotlib==3.9.1 \
    fvcore iopath pycocotools omegaconf hydra-core termcolor yacs tabulate cloudpickle

!{PY} -c "import sys, torch; print('venv python:', sys.version); print('torch:', torch.__version__, torch.cuda.is_available())"

## 4. 추론 스크립트

앱 서버를 붙일 때 `try_on()`을 그대로 재사용한다.

**입력**: 인물 이미지 경로, 옷 이미지 경로, 옷 종류(`upper`/`lower`/`overall`/`inner`/`outer`)
**출력**: 합성 이미지

In [ ]:
worker = r'''
import os, sys, glob, argparse, torch
REPO_DIR = '/content/scratch/CatVTON'
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor
from model.pipeline import CatVTONPipeline
from model.cloth_masker import AutoMasker, vis_mask
from utils import init_weight_dtype, resize_and_crop, resize_and_padding
from PIL import Image

WIDTH, HEIGHT = 768, 1024
repo_path = snapshot_download(repo_id='zhengchong/CatVTON')

pipeline = CatVTONPipeline(
    base_ckpt='runwayml/stable-diffusion-inpainting',
    attn_ckpt=repo_path,
    attn_ckpt_version='mix',
    weight_dtype=init_weight_dtype('fp16'),
    use_tf32=True,
    device='cuda',
    skip_safety_check=True,   # 신버전 transformers와 API 불일치 회피
)
automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, 'DensePose'),
    schp_ckpt=os.path.join(repo_path, 'SCHP'),
    device='cuda',
)
mask_processor = VaeImageProcessor(vae_scale_factor=8, do_normalize=False,
                                   do_binarize=True, do_convert_grayscale=True)

def try_on(person_path, garment_path, cloth_type='upper', steps=30, guidance_scale=2.5, seed=42):
    person = resize_and_crop(Image.open(person_path).convert('RGB'), (WIDTH, HEIGHT))
    garment = resize_and_padding(Image.open(garment_path).convert('RGB'), (WIDTH, HEIGHT))
    mask = automasker(person, cloth_type)['mask']
    mask = mask_processor.blur(mask, blur_factor=9)
    result = pipeline(
        image=person, condition_image=garment, mask=mask,
        num_inference_steps=steps, guidance_scale=guidance_scale,
        generator=torch.Generator(device='cuda').manual_seed(seed),
    )[0]
    return result, person, garment, mask

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--person'); ap.add_argument('--garment')
    ap.add_argument('--cloth-type', default='upper')
    ap.add_argument('--steps', type=int, default=30)
    ap.add_argument('--out', default='/content/outputs')
    a = ap.parse_args()

    person_path = a.person or sorted(glob.glob('resource/demo/example/person/men/*'))[0]
    garment_path = a.garment or sorted(glob.glob('resource/demo/example/condition/upper/*'))[0]
    print('person:', person_path, '| garment:', garment_path, flush=True)

    result, person, garment, mask = try_on(person_path, garment_path, a.cloth_type, a.steps)
    os.makedirs(a.out, exist_ok=True)
    result.save(a.out + '/result.png')
    person.save(a.out + '/person.png')
    garment.save(a.out + '/garment.png')
    vis_mask(person, mask).save(a.out + '/mask.png')
    print('DONE ->', a.out, flush=True)
'''
with open('/content/tryon_worker.py', 'w') as f:
    f.write(worker)
print('written: /content/tryon_worker.py')

## 5. 실행

인자 없이 돌리면 저장소 데모 이미지를 쓴다.
본인 사진으로 하려면 `--person`, `--garment`에 경로를 넘긴다.

In [ ]:
!{PY} /content/tryon_worker.py --out /content/outputs

In [ ]:
from IPython.display import display
from PIL import Image

for name in ('person', 'garment', 'mask', 'result'):
    p = f'/content/outputs/{name}.png'
    if os.path.exists(p):
        print(name)
        display(Image.open(p).resize((288, 384)))

## 6. 실패 케이스 수집

계획서의 우려사항(측면 사진, 팔로 옷 가림, 특이 포즈)을 의도적으로 넣어
어떤 입력에서 품질이 무너지는지 기록한다. 중간발표 자료에 그대로 쓸 수 있다.

In [ ]:
import glob

persons = sorted(glob.glob('/content/scratch/CatVTON/resource/demo/example/person/men/*'))[:2]
garments = sorted(glob.glob('/content/scratch/CatVTON/resource/demo/example/condition/upper/*'))[:2]

for i, p in enumerate(persons):
    for j, g in enumerate(garments):
        out = f'/content/outputs/p{i}_g{j}'
        !{PY} /content/tryon_worker.py --person "{p}" --garment "{g}" --out "{out}"
        display(Image.open(out + '/result.png').resize((288, 384)))